# Analyze Safety–Capability Tradeoff

Load all evaluation results and analyze:
- Pareto frontier: refusal quality vs capability
- Over-refusal tradeoff: XSTest safe false refusal rate vs unsafe refusal rate
- Capability tax per benchmark
- Protected interference energy vs capability tax correlation

Run after evaluation scripts (02, 03, 07) have produced result JSONs.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.reporting.make_tables import build_main_table, build_ablation_table
from src.reporting.make_figures import (
    plot_pareto_frontier, plot_overrefusal_tradeoff, plot_capability_tax_bars
)

RESULTS_DIR = ROOT / 'results/logs'
print('Results dir:', RESULTS_DIR)
result_files = sorted(RESULTS_DIR.glob('*.json'))
print(f'Found {len(result_files)} result files:')
for f in result_files:
    print(' ', f.name)

## 1. Main Results Table

In [ ]:
if result_files:
    df = build_main_table(
        result_files=[str(f) for f in result_files]
    )
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 140)
    display(df)
else:
    print('No result files found. Run evaluation scripts first.')

## 2. Pareto Frontier

In [ ]:
if result_files:
    results = [json.loads(f.read_text()) for f in result_files]
    names = [f.stem for f in result_files]
    
    fig = plot_pareto_frontier(
        results=results,
        method_names=names,
        annotate=True,
    )
    plt.show()

## 3. Over-Refusal Tradeoff (XSTest)

In [ ]:
if result_files:
    fig = plot_overrefusal_tradeoff(
        results=results,
        method_names=names,
        annotate=True,
    )
    plt.show()

## 4. Capability Tax by Benchmark

In [ ]:
if result_files:
    fig = plot_capability_tax_bars(
        results=results,
        method_names=names,
    )
    plt.show()

## 5. Lambda Sweep Analysis (PF-LoRA)

In [ ]:
# Filter to PF-LoRA ablation results with different lambda values
pf_files = [f for f in result_files if 'protected_lora' in f.stem and 'lambda' in f.stem]

if len(pf_files) < 2:
    print('Run multiple lambda values (script 06) before this analysis.')
else:
    pf_results = [json.loads(f.read_text()) for f in pf_files]
    
    rows = []
    for result, fpath in zip(pf_results, pf_files):
        s = result.get('summary', {})
        # Extract lambda from filename: protected_lora_llama31_lambda1e-3_k64
        parts = fpath.stem.split('_')
        lambda_val = next((p.replace('lambda', '') for p in parts if p.startswith('lambda')), '?')
        k_val = next((p.replace('k', '') for p in parts if p.startswith('k') and p[1:].isdigit()), '?')
        
        rows.append({
            'lambda': lambda_val,
            'k': k_val,
            'unsafe_refusal_rate': s.get('unsafe_refusal_rate'),
            'safe_false_refusal_rate': s.get('safe_false_refusal_rate'),
            'avg_capability_score': s.get('avg_capability_score'),
            'capability_tax_avg': s.get('capability_tax_avg'),
        })
    
    df_sweep = pd.DataFrame(rows)
    print('Lambda sweep results:')
    display(df_sweep)
    
    # Plot: lambda vs capability tax
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    for ax, y_col, ylabel in [
        (axes[0], 'avg_capability_score', 'Avg Capability Score'),
        (axes[1], 'capability_tax_avg', 'Capability Tax (%) ↓'),
    ]:
        ax.plot(df_sweep['lambda'], df_sweep[y_col], marker='o', color='crimson')
        ax.set_xlabel('Lambda (protection strength)')
        ax.set_ylabel(ylabel)
        ax.set_title(f'PF-LoRA: Lambda vs {ylabel}')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. Summary Statistics

In [ ]:
if result_files and len(results) >= 2:
    # Find base model and best PF-LoRA results
    base_result = next((r for r, n in zip(results, names) if 'base' in n.lower()), None)
    pf_result = next((r for r, n in zip(results, names) if 'protected' in n.lower()), None)
    lora_result = next((r for r, n in zip(results, names) if 'baseline' in n.lower()), None)

    print('=== Summary Comparison ===')
    print(f'{'Metric':<35} | {'Base':>10} | {'LoRA':>10} | {'PF-LoRA':>10}')
    print('-' * 72)
    
    metrics = [
        ('unsafe_refusal_rate', 'Unsafe Refusal Rate'),
        ('safe_false_refusal_rate', 'Safe False Refusal Rate'),
        ('refusal_f1_wildguard', 'Refusal F1 (WildGuard)'),
        ('avg_capability_score', 'Avg Capability Score'),
        ('capability_tax_avg', 'Capability Tax (avg %)'),
    ]
    
    def get_val(result, key):
        if result is None:
            return '-'
        v = result.get('summary', {}).get(key)
        return f'{v:.4f}' if v is not None else '-'
    
    for key, label in metrics:
        base_v = get_val(base_result, key)
        lora_v = get_val(lora_result, key)
        pf_v = get_val(pf_result, key)
        print(f'{label:<35} | {base_v:>10} | {lora_v:>10} | {pf_v:>10}')